# Data Collection & Cleaning

Combines four quarterly BTS DB1B Market files (Q3 2024 – Q2 2025), removes incomplete records, and joins airport lat/lon from OurAirports for use as GNN node features.

In [2]:
import pandas as pd
import numpy as np
import os

## 1. Preview Raw Data

In [3]:
sample = pd.read_csv('../../data/rawdata/2024_q3.csv', low_memory=False)
print(f"Shape: {sample.shape}")
print(f"Columns: {list(sample.columns)}")
sample.head()

Shape: (8297869, 15)
Columns: ['YEAR', 'QUARTER', 'ORIGIN_AIRPORT_ID', 'ORIGIN', 'DEST_AIRPORT_ID', 'DEST', 'REPORTING_CARRIER', 'TICKET_CARRIER', 'OPERATING_CARRIER', 'BULK_FARE', 'PASSENGERS', 'MARKET_FARE', 'MARKET_DISTANCE', 'NONSTOP_MILES', 'MKT_GEO_TYPE']


,YEAR,QUARTER,ORIGIN_AIRPORT_ID,ORIGIN,DEST_AIRPORT_ID,DEST,REPORTING_CARRIER,TICKET_CARRIER,OPERATING_CARRIER,BULK_FARE,PASSENGERS,MARKET_FARE,MARKET_DISTANCE,NONSTOP_MILES,MKT_GEO_TYPE
0,2024,3,10135,ABE,10136,ABI,MQ,AA,99,0.0,1.0,430.5,1575.0,1458.0,2
1,2024,3,10135,ABE,10140,ABQ,9E,DL,99,0.0,1.0,5.5,1961.0,1738.0,2
2,2024,3,10135,ABE,10140,ABQ,9E,DL,99,0.0,1.0,148.5,1961.0,1738.0,2
3,2024,3,10135,ABE,10140,ABQ,9E,DL,99,0.0,1.0,182.0,1961.0,1738.0,2
4,2024,3,10135,ABE,10140,ABQ,9E,DL,99,0.0,1.0,309.5,1961.0,1738.0,2


## 2. Combine Quarterly Files

In [4]:
files = [
    '../../data/rawdata/2024_q3.csv',
    '../../data/rawdata/2024_q4.csv',
    '../../data/rawdata/2025_q1.csv',
    '../../data/rawdata/2025_q2.csv',
]

dfs = []
for path in files:
    df = pd.read_csv(path, low_memory=False)
    print(f"  {os.path.basename(path)}: {df.shape}")
    dfs.append(df)

combined = pd.concat(dfs, ignore_index=True)
print(f"\nCombined shape: {combined.shape}")
print(f"Quarters present: {sorted(combined[['YEAR','QUARTER']].drop_duplicates().apply(tuple, axis=1).tolist())}")
combined.head()

  2024_q3.csv: (8297869, 15)
  2024_q4.csv: (8525077, 15)
  2025_q1.csv: (7297028, 15)
  2025_q2.csv: (8450420, 15)

Combined shape: (32570394, 15)
Quarters present: [(2024, 3), (2024, 4), (2025, 1), (2025, 2)]


,YEAR,QUARTER,ORIGIN_AIRPORT_ID,ORIGIN,DEST_AIRPORT_ID,DEST,REPORTING_CARRIER,TICKET_CARRIER,OPERATING_CARRIER,BULK_FARE,PASSENGERS,MARKET_FARE,MARKET_DISTANCE,NONSTOP_MILES,MKT_GEO_TYPE
0,2024,3,10135,ABE,10136,ABI,MQ,AA,99,0.0,1.0,430.5,1575.0,1458.0,2
1,2024,3,10135,ABE,10140,ABQ,9E,DL,99,0.0,1.0,5.5,1961.0,1738.0,2
2,2024,3,10135,ABE,10140,ABQ,9E,DL,99,0.0,1.0,148.5,1961.0,1738.0,2
3,2024,3,10135,ABE,10140,ABQ,9E,DL,99,0.0,1.0,182.0,1961.0,1738.0,2
4,2024,3,10135,ABE,10140,ABQ,9E,DL,99,0.0,1.0,309.5,1961.0,1738.0,2


## 3. Missing Value Analysis

In [4]:
print("=== Missing Value Counts ===")
print(combined.isna().sum())
print(f"\nRows with any missing value: {combined.isna().any(axis=1).sum():,}  ({combined.isna().any(axis=1).mean():.1%} of total)")

=== Missing Value Counts ===
YEAR                 0
QUARTER              0
ORIGIN_AIRPORT_ID    0
ORIGIN               0
DEST_AIRPORT_ID      0
DEST                 0
REPORTING_CARRIER    0
TICKET_CARRIER       0
OPERATING_CARRIER    0
BULK_FARE            0
PASSENGERS           0
MARKET_FARE          0
MARKET_DISTANCE      0
NONSTOP_MILES        0
MKT_GEO_TYPE         0
dtype: int64

Rows with any missing value: 0  (0.0% of total)


## 4. Clean Data

Drop rows with missing values and remove zero/negative fares (data entry artifacts).

In [5]:
df_clean = (
    combined
    .dropna()
    .query("MARKET_FARE > 0")
    .reset_index(drop=True)
)

print(f"Before cleaning: {combined.shape[0]:,} rows")
print(f"After cleaning:  {df_clean.shape[0]:,} rows")
print(f"Removed:         {combined.shape[0] - df_clean.shape[0]:,} rows ({(combined.shape[0] - df_clean.shape[0]) / combined.shape[0]:.1%})")
df_clean.describe()

Before cleaning: 32,570,394 rows
After cleaning:  32,541,848 rows
Removed:         28,546 rows (0.1%)


,YEAR,QUARTER,ORIGIN_AIRPORT_ID,DEST_AIRPORT_ID,BULK_FARE,PASSENGERS,MARKET_FARE,MARKET_DISTANCE,NONSTOP_MILES,MKT_GEO_TYPE
count,3.254185e+07,3.254185e+07,3.254185e+07,3.254185e+07,32541848.0,3.254185e+07,3.254185e+07,3.254185e+07,3.254185e+07,3.254185e+07
mean,2.024483e+03,2.554236e+00,1.282923e+04,1.282866e+04,0.0,1.895133e+00,2.710958e+02,1.301878e+03,1.221670e+03,1.932522e+00
std,4.997272e-01,1.103920e+00,1.541862e+03,1.542257e+03,0.0,4.971489e+00,2.216202e+02,8.166206e+02,7.736060e+02,2.508474e-01
min,2.024000e+03,1.000000e+00,1.013500e+04,1.013500e+04,0.0,1.000000e+00,1.100000e-01,1.100000e+01,1.100000e+01,1.000000e+00
25%,2.024000e+03,2.000000e+00,1.129800e+04,1.129800e+04,0.0,1.000000e+00,1.490000e+02,7.100000e+02,6.540000e+02,2.000000e+00
50%,2.024000e+03,3.000000e+00,1.294500e+04,1.294500e+04,0.0,1.000000e+00,2.310000e+02,1.087000e+03,1.020000e+03,2.000000e+00
75%,2.025000e+03,4.000000e+00,1.410700e+04,1.410700e+04,0.0,1.000000e+00,3.400000e+02,1.751000e+03,1.643000e+03,2.000000e+00
max,2.025000e+03,4.000000e+00,1.686900e+04,1.686900e+04,0.0,1.032000e+03,4.443200e+04,1.134400e+04,9.411000e+03,2.000000e+00


# realized that there are still abnormally low fares
Seached on google and various sites and found the cheapest flight is $19 from pittsburg to dubois, so I am going to filter that out now

In [6]:
df_clean[df_clean['MARKET_FARE'] < 40]

,YEAR,QUARTER,ORIGIN_AIRPORT_ID,ORIGIN,DEST_AIRPORT_ID,DEST,REPORTING_CARRIER,TICKET_CARRIER,OPERATING_CARRIER,BULK_FARE,PASSENGERS,MARKET_FARE,MARKET_DISTANCE,NONSTOP_MILES,MKT_GEO_TYPE
1,2024,3,10135,ABE,10140,ABQ,9E,DL,99,0.0,1.0,5.50,1961.0,1738.0,2
39,2024,3,10135,ABE,10157,ACV,G7,UA,99,0.0,1.0,5.00,2976.0,2518.0,2
40,2024,3,10135,ABE,10157,ACV,OO,99,99,0.0,1.0,5.38,2698.0,2518.0,2
75,2024,3,10135,ABE,10397,ATL,9E,DL,9E,0.0,1.0,2.86,692.0,692.0,2
76,2024,3,10135,ABE,10397,ATL,9E,DL,9E,0.0,1.0,5.50,692.0,692.0,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32541567,2025,2,16869,XWA,14747,SEA,OO,UA,99,0.0,1.0,5.00,1606.0,863.0,2
32541683,2025,2,16869,XWA,14869,SLC,UA,UA,99,0.0,1.0,5.50,973.0,656.0,2
32541726,2025,2,16869,XWA,15016,STL,OO,DL,99,0.0,1.0,5.50,1001.0,937.0,2
32541776,2025,2,16869,XWA,15370,TUL,OO,UA,OO,0.0,1.0,5.50,1123.0,924.0,2


In [7]:
df_clean = (
    combined
    .dropna()
    .query("MARKET_FARE > 19")
    .reset_index(drop=True)
)

print(f"Before cleaning: {combined.shape[0]:,} rows")
print(f"After cleaning:  {df_clean.shape[0]:,} rows")
print(f"Removed:         {combined.shape[0] - df_clean.shape[0]:,} rows ({(combined.shape[0] - df_clean.shape[0]) / combined.shape[0]:.1%})")
df_clean.describe()

Before cleaning: 32,570,394 rows
After cleaning:  30,976,039 rows
Removed:         1,594,355 rows (4.9%)


,YEAR,QUARTER,ORIGIN_AIRPORT_ID,DEST_AIRPORT_ID,BULK_FARE,PASSENGERS,MARKET_FARE,MARKET_DISTANCE,NONSTOP_MILES,MKT_GEO_TYPE
count,3.097604e+07,3.097604e+07,3.097604e+07,3.097604e+07,30976039.0,3.097604e+07,3.097604e+07,3.097604e+07,3.097604e+07,3.097604e+07
mean,2.024483e+03,2.556007e+00,1.282755e+04,1.282683e+04,0.0,1.800310e+00,2.845049e+02,1.294039e+03,1.216806e+03,1.932594e+00
std,4.996969e-01,1.103854e+00,1.540203e+03,1.540644e+03,0.0,3.411450e+00,2.187720e+02,8.137536e+02,7.715996e+02,2.507233e-01
min,2.024000e+03,1.000000e+00,1.013500e+04,1.013500e+04,0.0,1.000000e+00,1.901000e+01,1.100000e+01,1.100000e+01,1.000000e+00
25%,2.024000e+03,2.000000e+00,1.129800e+04,1.129800e+04,0.0,1.000000e+00,1.615000e+02,7.010000e+02,6.510000e+02,2.000000e+00
50%,2.024000e+03,3.000000e+00,1.291500e+04,1.291500e+04,0.0,1.000000e+00,2.390000e+02,1.080000e+03,1.015000e+03,2.000000e+00
75%,2.025000e+03,4.000000e+00,1.410700e+04,1.410700e+04,0.0,1.000000e+00,3.475000e+02,1.744000e+03,1.633000e+03,2.000000e+00
max,2.025000e+03,4.000000e+00,1.686900e+04,1.686900e+04,0.0,4.930000e+02,4.443200e+04,1.134400e+04,9.362000e+03,2.000000e+00


In [8]:
df_clean[df_clean['MARKET_FARE'] <40].sort_values('MARKET_DISTANCE', ascending=True).head(1000)

,YEAR,QUARTER,ORIGIN_AIRPORT_ID,ORIGIN,DEST_AIRPORT_ID,DEST,REPORTING_CARRIER,TICKET_CARRIER,OPERATING_CARRIER,BULK_FARE,PASSENGERS,MARKET_FARE,MARKET_DISTANCE,NONSTOP_MILES,MKT_GEO_TYPE
30948834,2025,2,15841,WRG,14256,PSG,AS,AS,AS,0.0,1.0,31.00,31.0,31.0,1
25798754,2025,2,11997,GST,12523,JNU,AS,AS,AS,0.0,1.0,20.68,41.0,41.0,1
2876699,2024,3,11997,GST,12523,JNU,AS,AS,AS,0.0,1.0,21.89,41.0,41.0,1
25798755,2025,2,11997,GST,12523,JNU,AS,AS,AS,0.0,1.0,34.66,41.0,41.0,1
2876703,2024,3,11997,GST,12523,JNU,AS,AS,AS,0.0,1.0,37.42,41.0,41.0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7611820,2024,3,15027,STX,14843,SJU,3M,3M,3M,0.0,1.0,36.00,94.0,94.0,1
7611821,2024,3,15027,STX,14843,SJU,3M,3M,3M,0.0,1.0,36.50,94.0,94.0,1
7611822,2024,3,15027,STX,14843,SJU,3M,3M,3M,0.0,1.0,38.00,94.0,94.0,1
22684782,2025,1,15027,STX,14843,SJU,B6,B6,B6,0.0,1.0,26.50,94.0,94.0,1


In [9]:
#drop abnormally low price for longer distances
df_clean = (
    combined
    .dropna()
    .query("MARKET_FARE > 19")
    .query("not (MARKET_DISTANCE > 1000 and MARKET_FARE < 100)")
    .reset_index(drop=True)
)

print(f"Before cleaning: {combined.shape[0]:,} rows")
print(f"After cleaning:  {df_clean.shape[0]:,} rows")
print(f"Removed:         {combined.shape[0] - df_clean.shape[0]:,} rows ({(combined.shape[0] - df_clean.shape[0]) / combined.shape[0]:.1%})")
df_clean.describe()

Before cleaning: 32,570,394 rows
After cleaning:  30,021,054 rows
Removed:         2,549,340 rows (7.8%)


,YEAR,QUARTER,ORIGIN_AIRPORT_ID,DEST_AIRPORT_ID,BULK_FARE,PASSENGERS,MARKET_FARE,MARKET_DISTANCE,NONSTOP_MILES,MKT_GEO_TYPE
count,3.002105e+07,3.002105e+07,3.002105e+07,3.002105e+07,30021054.0,3.002105e+07,3.002105e+07,3.002105e+07,3.002105e+07,3.002105e+07
mean,2.024483e+03,2.556529e+00,1.282577e+04,1.282496e+04,0.0,1.763401e+00,2.913097e+02,1.288264e+03,1.210881e+03,1.932867e+00
std,4.997094e-01,1.103584e+00,1.542402e+03,1.542900e+03,0.0,3.305224e+00,2.187856e+02,8.205366e+02,7.782231e+02,2.502517e-01
min,2.024000e+03,1.000000e+00,1.013500e+04,1.013500e+04,0.0,1.000000e+00,1.901000e+01,1.100000e+01,1.100000e+01,1.000000e+00
25%,2.024000e+03,2.000000e+00,1.129800e+04,1.129800e+04,0.0,1.000000e+00,1.688000e+02,6.890000e+02,6.410000e+02,2.000000e+00
50%,2.024000e+03,3.000000e+00,1.291700e+04,1.291700e+04,0.0,1.000000e+00,2.440000e+02,1.067000e+03,9.980000e+02,2.000000e+00
75%,2.025000e+03,4.000000e+00,1.410700e+04,1.410700e+04,0.0,1.000000e+00,3.520000e+02,1.745000e+03,1.635000e+03,2.000000e+00
max,2.025000e+03,4.000000e+00,1.686900e+04,1.686900e+04,0.0,4.930000e+02,4.443200e+04,1.134400e+04,9.362000e+03,2.000000e+00


In [10]:
# check again
df_clean[df_clean['MARKET_FARE'] <100].sort_values('MARKET_DISTANCE', ascending=False).head(100)

,YEAR,QUARTER,ORIGIN_AIRPORT_ID,ORIGIN,DEST_AIRPORT_ID,DEST,REPORTING_CARRIER,TICKET_CARRIER,OPERATING_CARRIER,BULK_FARE,PASSENGERS,MARKET_FARE,MARKET_DISTANCE,NONSTOP_MILES,MKT_GEO_TYPE
2720962,2024,3,11986,GRR,10994,CHS,WN,WN,WN,0.0,2.0,79.7,1000.0,750.0,2
3860798,2024,3,12898,LBE,11697,FLL,NK,NK,NK,0.0,1.0,56.0,1000.0,980.0,2
3860799,2024,3,12898,LBE,11697,FLL,NK,NK,NK,0.0,1.0,74.0,1000.0,980.0,2
3860800,2024,3,12898,LBE,11697,FLL,NK,NK,NK,0.0,1.0,76.0,1000.0,980.0,2
3860801,2024,3,12898,LBE,11697,FLL,NK,NK,NK,0.0,1.0,86.0,1000.0,980.0,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29993695,2025,2,15628,VRB,12197,HPN,MX,MX,MX,0.0,4.0,65.5,1000.0,1000.0,2
29993696,2025,2,15628,VRB,12197,HPN,MX,MX,MX,0.0,4.0,85.0,1000.0,1000.0,2
29993706,2025,2,15628,VRB,12197,HPN,MX,MX,MX,0.0,5.0,85.0,1000.0,1000.0,2
29993707,2025,2,15628,VRB,12197,HPN,MX,MX,MX,0.0,5.0,92.0,1000.0,1000.0,2


In [6]:
#realized that there are still some pretty low fares for longer distances
df_clean = (
    combined
    .dropna()
    .query("MARKET_FARE > 19")
    # Filter out flights where price per mile is less than 10 cents
    .query("MARKET_FARE / MARKET_DISTANCE >= 0.10")
    .query("not (MARKET_DISTANCE > 1000 and MARKET_FARE < 100)")
    .reset_index(drop=True)
)

print(f"Before cleaning: {combined.shape[0]:,} rows")
print(f"After cleaning:  {df_clean.shape[0]:,} rows")
print(f"Removed:         {combined.shape[0] - df_clean.shape[0]:,} rows ({(combined.shape[0] - df_clean.shape[0]) / combined.shape[0]:.1%})")
df_clean.describe()

Before cleaning: 32,570,394 rows
After cleaning:  26,474,530 rows
Removed:         6,095,864 rows (18.7%)


,YEAR,QUARTER,ORIGIN_AIRPORT_ID,DEST_AIRPORT_ID,BULK_FARE,PASSENGERS,MARKET_FARE,MARKET_DISTANCE,NONSTOP_MILES,MKT_GEO_TYPE
count,2.647453e+07,2.647453e+07,2.647453e+07,2.647453e+07,26474530.0,2.647453e+07,2.647453e+07,2.647453e+07,2.647453e+07,2.647453e+07
mean,2.024483e+03,2.558598e+00,1.280781e+04,1.280669e+04,0.0,1.704213e+00,3.095792e+02,1.190367e+03,1.119524e+03,1.947012e+00
std,4.997227e-01,1.105994e+00,1.547319e+03,1.547901e+03,0.0,3.164895e+00,2.248992e+02,7.428078e+02,7.064374e+02,2.240103e-01
min,2.024000e+03,1.000000e+00,1.013500e+04,1.013500e+04,0.0,1.000000e+00,1.901000e+01,1.100000e+01,1.100000e+01,1.000000e+00
25%,2.024000e+03,2.000000e+00,1.129200e+04,1.129200e+04,0.0,1.000000e+00,1.835000e+02,6.510000e+02,6.110000e+02,2.000000e+00
50%,2.024000e+03,3.000000e+00,1.289600e+04,1.289600e+04,0.0,1.000000e+00,2.620000e+02,1.013000e+03,9.510000e+02,2.000000e+00
75%,2.025000e+03,4.000000e+00,1.410700e+04,1.410700e+04,0.0,1.000000e+00,3.690000e+02,1.574000e+03,1.481000e+03,2.000000e+00
max,2.025000e+03,4.000000e+00,1.686900e+04,1.686900e+04,0.0,4.930000e+02,4.443200e+04,1.134400e+04,9.362000e+03,2.000000e+00


In [12]:
# check again
df_clean[df_clean['MARKET_FARE'] <100].sort_values('MARKET_DISTANCE', ascending=False).head(100)

,YEAR,QUARTER,ORIGIN_AIRPORT_ID,ORIGIN,DEST_AIRPORT_ID,DEST,REPORTING_CARRIER,TICKET_CARRIER,OPERATING_CARRIER,BULK_FARE,PASSENGERS,MARKET_FARE,MARKET_DISTANCE,NONSTOP_MILES,MKT_GEO_TYPE
8778322,2024,4,11540,ELP,14771,SFO,WN,WN,WN,0.0,1.0,99.95,997.0,993.0,2
26299015,2025,2,15304,TPA,13232,MDW,F9,F9,F9,0.0,1.0,99.93,997.0,997.0,2
19438533,2025,1,15304,TPA,13232,MDW,WN,WN,WN,0.0,1.0,99.91,997.0,997.0,2
11712968,2024,4,14057,PDX,14679,SAN,UA,UA,UA,0.0,1.0,99.70,997.0,933.0,2
21793490,2025,2,11618,EWR,15304,TPA,B6,B6,B6,0.0,1.0,99.85,997.0,997.0,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17851189,2025,1,14057,PDX,11292,DEN,WN,WN,WN,0.0,1.0,99.50,991.0,991.0,2
8387256,2024,4,11292,DEN,14057,PDX,WN,WN,WN,0.0,1.0,99.50,991.0,991.0,2
8385707,2024,4,11292,DEN,14057,PDX,AS,AS,AS,0.0,1.0,99.50,991.0,991.0,2
15057662,2025,1,11292,DEN,14057,PDX,WN,WN,WN,0.0,1.0,99.50,991.0,991.0,2


In [ ]:
#check high values, and realized that super expensive flights for certain routes were significantly impacting the overall pricing structure, no reason for flight from minessota to chicago to be 40k USD, even if it was first class
df_clean.sort_values('MARKET_FARE', ascending=False).head(20)

,YEAR,QUARTER,ORIGIN_AIRPORT_ID,ORIGIN,DEST_AIRPORT_ID,DEST,REPORTING_CARRIER,TICKET_CARRIER,OPERATING_CARRIER,BULK_FARE,PASSENGERS,MARKET_FARE,MARKET_DISTANCE,NONSTOP_MILES,MKT_GEO_TYPE
9234232,2024,4,12129,HIB,13930,ORD,OO,DL,99,0.0,1.0,44432.0,508.0,445.0,2
22097069,2025,2,12129,HIB,11057,CLT,OO,DL,99,0.0,1.0,38735.0,1104.0,1041.0,2
17380537,2025,1,13487,MSP,16869,XWA,OO,DL,OO,0.0,1.0,30656.0,553.0,553.0,2
13677829,2024,4,16869,XWA,14771,SFO,OO,DL,99,0.0,1.0,23180.0,2142.0,1191.0,2
15796915,2025,1,12129,HIB,13487,MSP,OO,DL,OO,0.0,1.0,19653.0,174.0,174.0,2
22097189,2025,2,12129,HIB,13487,MSP,OO,DL,OO,0.0,1.0,19412.0,174.0,174.0,2
9234218,2024,4,12129,HIB,13487,MSP,OO,DL,OO,0.0,1.0,18338.0,174.0,174.0,2
23685099,2025,2,13303,MIA,12953,LGA,AA,AA,99,0.0,1.0,16936.0,1133.0,1096.0,2
23420436,2025,2,13204,MCO,12892,LAX,AA,AA,AA,0.0,1.0,16936.0,2534.0,2218.0,2
25122408,2025,2,14570,RNO,11697,FLL,AA,AA,AA,0.0,1.0,16936.0,2464.0,2461.0,2


In [13]:
df_clean = (
    combined
    .dropna()
    .query("MARKET_FARE > 19")
    .query("MARKET_FARE / MARKET_DISTANCE >= 0.10")
    .query("not (MARKET_DISTANCE > 1000 and MARKET_FARE < 100)")
    .reset_index(drop=True)
)

# IQR outlier removal on MARKET_FARE
Q1 = df_clean['MARKET_FARE'].quantile(0.25)
Q3 = df_clean['MARKET_FARE'].quantile(0.75)
IQR = Q3 - Q1
df_clean = df_clean[
    (df_clean['MARKET_FARE'] >= Q1 - 1.5 * IQR) &
    (df_clean['MARKET_FARE'] <= Q3 + 1.5 * IQR)
]

print(f"Before cleaning: {combined.shape[0]:,} rows")
print(f"After cleaning:  {df_clean.shape[0]:,} rows")
print(f"Removed:         {combined.shape[0] - df_clean.shape[0]:,} rows ({(combined.shape[0] - df_clean.shape[0]) / combined.shape[0]:.1%})")
df_clean.describe()

Before cleaning: 32,570,394 rows
After cleaning:  25,090,503 rows
Removed:         7,479,891 rows (23.0%)


,YEAR,QUARTER,ORIGIN_AIRPORT_ID,DEST_AIRPORT_ID,BULK_FARE,PASSENGERS,MARKET_FARE,MARKET_DISTANCE,NONSTOP_MILES,MKT_GEO_TYPE
count,2.509050e+07,2.509050e+07,2.509050e+07,2.509050e+07,25090503.0,2.509050e+07,2.509050e+07,2.509050e+07,2.509050e+07,2.509050e+07
mean,2.024483e+03,2.559572e+00,1.280464e+04,1.280403e+04,0.0,1.735808e+00,2.727143e+02,1.140470e+03,1.072343e+03,1.954937e+00
std,4.996997e-01,1.104007e+00,1.550097e+03,1.550544e+03,0.0,3.241072e+00,1.237165e+02,6.842711e+02,6.501630e+02,2.074421e-01
min,2.024000e+03,1.000000e+00,1.013500e+04,1.013500e+04,0.0,1.000000e+00,1.901000e+01,1.100000e+01,1.100000e+01,1.000000e+00
25%,2.024000e+03,2.000000e+00,1.129200e+04,1.129200e+04,0.0,1.000000e+00,1.795000e+02,6.370000e+02,5.990000e+02,2.000000e+00
50%,2.024000e+03,3.000000e+00,1.294500e+04,1.294500e+04,0.0,1.000000e+00,2.530000e+02,9.890000e+02,9.290000e+02,2.000000e+00
75%,2.025000e+03,4.000000e+00,1.410700e+04,1.410700e+04,0.0,1.000000e+00,3.470000e+02,1.502000e+03,1.415000e+03,2.000000e+00
max,2.025000e+03,4.000000e+00,1.686900e+04,1.686900e+04,0.0,4.930000e+02,6.472500e+02,6.383000e+03,5.981000e+03,2.000000e+00


In [15]:
# the fares make more sense now
df_clean.sort_values('MARKET_FARE', ascending=False).head(20)

,YEAR,QUARTER,ORIGIN_AIRPORT_ID,ORIGIN,DEST_AIRPORT_ID,DEST,REPORTING_CARRIER,TICKET_CARRIER,OPERATING_CARRIER,BULK_FARE,PASSENGERS,MARKET_FARE,MARKET_DISTANCE,NONSTOP_MILES,MKT_GEO_TYPE
17756217,2025,1,13930,ORD,15024,STT,UA,UA,UA,0.0,1.0,647.25,2116.0,2116.0,1
26195539,2025,2,15024,STT,11278,DCA,YX,DL,DL,0.0,1.0,647.25,2146.0,1588.0,1
12881751,2024,4,14771,SFO,11697,FLL,AS,99,99,0.0,1.0,647.25,2623.0,2584.0,2
7443826,2024,4,10721,BOS,14747,SEA,AS,AS,AS,0.0,1.0,647.25,2496.0,2496.0,2
15800102,2025,1,12173,HNL,10397,ATL,UA,UA,UA,0.0,1.0,647.25,4537.0,4502.0,1
1833492,2024,3,11298,DFW,12982,LIH,AA,AA,99,0.0,1.0,647.25,3886.0,3847.0,1
12721678,2024,4,14747,SEA,10721,BOS,AS,AS,AS,0.0,1.0,647.25,2496.0,2496.0,2
2199638,2024,3,11618,EWR,14771,SFO,UA,UA,UA,0.0,1.0,647.25,2565.0,2565.0,2
25665718,2025,2,14771,SFO,11298,DFW,AA,AA,AA,0.0,1.0,647.25,1464.0,1464.0,2
13538057,2024,4,15304,TPA,14908,SNA,UA,99,99,0.0,1.0,647.25,2235.0,2127.0,2


## 5. Join Airport Coordinates

`airports.dat.txt` (OurAirports) provides latitude and longitude keyed on IATA code. We join on `ORIGIN` and `DEST` to add geo features needed by the GNN node embeddings.

In [16]:
airport_cols = ['airport_id', 'name', 'city', 'country', 'iata', 'icao',
                'latitude', 'longitude', 'altitude', 'timezone', 'dst', 'tz_db', 'type', 'source']

airports_raw = pd.read_csv('../../data/rawdata/airports.dat.txt', header=None, names=airport_cols)

# Keep only rows with a valid IATA code
coords = (
    airports_raw[airports_raw['iata'].notna() & (airports_raw['iata'] != '\\N')]
    [['iata', 'latitude', 'longitude']]
    .drop_duplicates('iata')
)

print(f"Airports with valid IATA codes: {len(coords):,}")
coords.head()

Airports with valid IATA codes: 6,072


,iata,latitude,longitude
0,GKA,-6.081690,145.391998
1,MAG,-5.207080,145.789001
2,HGU,-5.826790,144.296005
3,LAE,-6.569803,146.725977
4,POM,-9.443380,147.220001


In [17]:
df_geo = (
    df_clean
    .merge(
        coords.rename(columns={'iata': 'ORIGIN', 'latitude': 'ORIGIN_LAT', 'longitude': 'ORIGIN_LON'}),
        on='ORIGIN', how='inner'
    )
    .merge(
        coords.rename(columns={'iata': 'DEST', 'latitude': 'DEST_LAT', 'longitude': 'DEST_LON'}),
        on='DEST', how='inner'
    )
)

print(f"After geo join: {df_geo.shape[0]:,} rows, {df_geo.shape[1]} columns")
print(f"Rows lost in join (airports not in coords): {df_clean.shape[0] - df_geo.shape[0]:,}")
df_geo.head()

After geo join: 25,072,315 rows, 19 columns
Rows lost in join (airports not in coords): 18,188


,YEAR,QUARTER,ORIGIN_AIRPORT_ID,ORIGIN,DEST_AIRPORT_ID,DEST,REPORTING_CARRIER,TICKET_CARRIER,OPERATING_CARRIER,BULK_FARE,PASSENGERS,MARKET_FARE,MARKET_DISTANCE,NONSTOP_MILES,MKT_GEO_TYPE,ORIGIN_LAT,ORIGIN_LON,DEST_LAT,DEST_LON
0,2024,3,10135,ABE,10136,ABI,MQ,AA,99,0.0,1.0,430.5,1575.0,1458.0,2,40.6521,-75.440804,32.411301,-99.681900
1,2024,3,10135,ABE,10140,ABQ,9E,DL,99,0.0,1.0,309.5,1961.0,1738.0,2,40.6521,-75.440804,35.040199,-106.609001
2,2024,3,10135,ABE,10140,ABQ,9E,DL,99,0.0,1.0,324.5,1961.0,1738.0,2,40.6521,-75.440804,35.040199,-106.609001
3,2024,3,10135,ABE,10140,ABQ,9E,DL,99,0.0,1.0,378.5,1961.0,1738.0,2,40.6521,-75.440804,35.040199,-106.609001
4,2024,3,10135,ABE,10140,ABQ,9E,DL,99,0.0,1.0,381.8,2580.0,1738.0,2,40.6521,-75.440804,35.040199,-106.609001


In [18]:
print("=== Final Dataset Summary ===")
print(f"Shape: {df_geo.shape}")
print(f"\nMissing values: {df_geo.isna().sum().sum()}")
print(f"\nRow counts by quarter:")
print(df_geo.groupby(['YEAR', 'QUARTER']).size().to_string())
print(f"\nUnique origin airports:  {df_geo['ORIGIN'].nunique():,}")
print(f"Unique dest airports:    {df_geo['DEST'].nunique():,}")
print(f"Unique routes:           {df_geo.groupby(['ORIGIN', 'DEST']).ngroups:,}")
print(f"\nMARKET_FARE stats:")
print(df_geo['MARKET_FARE'].describe())

=== Final Dataset Summary ===
Shape: (25072315, 19)

Missing values: 0

Row counts by quarter:
YEAR  QUARTER
2024  3          6346126
      4          6624649
2025  1          5565168
      2          6536372

Unique origin airports:  447
Unique dest airports:    449
Unique routes:           81,857

MARKET_FARE stats:
count    2.507232e+07
mean     2.726650e+02
std      1.236967e+02
min      1.901000e+01
25%      1.795000e+02
50%      2.530000e+02
75%      3.470000e+02
max      6.472500e+02
Name: MARKET_FARE, dtype: float64


## 6. Save Clean Data

In [ ]:
out_path = '../../data/clean_data/T_DB1B_MARKET_CLEAN.csv'
df_geo.to_csv(out_path, index=False)
print(f"Saved {df_geo.shape[0]:,} rows × {df_geo.shape[1]} columns → {out_path}")